# 📊 Day 3 Homework Grading System: Evaluation & Optimization
## For instructors

---
### How to use
1. Run Setup once
2. Paste the GitHub link → Run grading → Results are automatically appended to CSV/JSON
3. Change the link → Grade the next person

> **🇹🇭 Thai Text in This Notebook**
>
> Sample data and queries are in Thai because this workshop teaches Thai RAG. English translations are provided as inline comments (`# "translation"`).

## 📦 Setup (Run once)

In [ ]:
%%time
!pip install -q google-genai requests nbformat

import hashlib, os, json, random, re, csv, requests
from datetime import datetime
import nbformat
from google import genai

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    GOOGLE_API_KEY = input('🔑 Gemini API Key: ')

client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL = 'gemini-2.5-pro'

SCORES_CSV = 'day3_scores.csv'
SCORES_JSON = 'day3_scores.json'

if not os.path.exists(SCORES_CSV):
    with open(SCORES_CSV, 'w', encoding='utf-8', newline='') as f:
        csv.writer(f).writerow(['Timestamp','Name','Student ID','Phone','LINE',
            'Step1','Step2','Step3','Step4','Total','AI Suspected','Feedback','URL'])
    print(f'📄 สร้าง {SCORES_CSV} ใหม่')  # "📄 Created new {SCORES_CSV}"
if not os.path.exists(SCORES_JSON):
    with open(SCORES_JSON, 'w') as f: json.dump([], f)

print(f'✅ Setup เรียบร้อย | Model: {MODEL}')  # "✅ Setup complete | Model: {MODEL}"

## 🔧 Grading functions (Run once)

In [ ]:
def generate_expected_day3(student_id):
    """Generate answer key from student ID"""
    seed = int(hashlib.md5(student_id.encode()).hexdigest(), 16) % (10**9)
    rng = random.Random(seed)
    all_topics = ['healthcare','banking','education','agriculture','logistics']
    topics = rng.sample(all_topics, 4)
    return {
        'topics': topics,
        'num_texts_per_topic': 2,
        'expected_tasks': [
            'RAG Pipeline (search_qdrant + rag_answer)',
            'LLM-as-Judge eval >= 5 questions',
            'Optimization Before/After >= 2 changes',
            'Agent + Test cases >= 5',
        ],
        'verification': hashlib.sha256(f'{student_id}_day3_hw'.encode()).hexdigest()[:12]
    }


def fetch_notebook(url):
    raw = url.replace('github.com','raw.githubusercontent.com').replace('/blob/','/')
    resp = requests.get(raw); resp.raise_for_status()
    nb = nbformat.reads(resp.text, as_version=4)
    info = {'student_name':'','student_id':'','phone':'','line_id':''}
    content = ''
    for i, cell in enumerate(nb.cells):
        if cell.cell_type == 'code':
            for key, var in [('student_id','STUDENT_ID'),('student_name','STUDENT_NAME'),
                             ('phone','PHONE'),('line_id','LINE_ID')]:
                m = re.search(rf"{var}\s*=\s*['\"](.*?)['\"]", cell.source)
                if m: info[key] = m.group(1)
        content += f'\n=== CELL {i} ({cell.cell_type.upper()}) ===\n{cell.source}'
        if hasattr(cell,'outputs'):
            for out in cell.outputs:
                if hasattr(out,'text'): content += f'\n--- OUTPUT ---\n{out.text}'
                elif hasattr(out,'data'):
                    for mime,data in out.data.items():
                        if 'text' in mime:
                            content += f'\n--- OUTPUT ---\n{data if isinstance(data,str) else chr(10).join(data)}'
    return info, content


def grade_day3(student_id, content, expected):
    prompt = f'''ตรวจการบ้าน Day 3 - Evaluation & Optimization\n\n
เฉลย: topics={expected["topics"]}, texts_per_topic={expected["num_texts_per_topic"]},
verification={expected["verification"]}\n\n
Notebook:\n{content[:15000]}\n\n
เกณฑ์:\n
Step 1 (3): 1-Chunk+Embed+Qdrant pipeline ทำงานได้, 1-search_qdrant() return results, 1-rag_answer() ตอบได้จริง\n
Step 2 (2.5): 1-eval questions >= 5 ข้อจากข้อมูลตัวเอง, 0.5-llm_judge ทำงานได้, 1-แสดงผลคะแนนเฉลี่ย\n
Step 3 (2.5): 1-ปรับปรุง >= 2 อย่าง (chunk_size/prompt/top_k), 1-มี Before/After comparison, 0.5-อธิบายเหตุผล\n
Step 4 (2): 1-Agent+Tool ทำงานได้, 0.5-test cases >= 5, 0.5-มี pass rate\n\n
ตอบ JSON: {{"step1_score":0,"step1_feedback":"...","step2_score":0,"step2_feedback":"...",
"step3_score":0,"step3_feedback":"...","step4_score":0,"step4_feedback":"...",
"total_score":0,"overall_feedback":"...","is_ai_suspected":false,"ai_reason":"..."}}
feedback in Thai, if there is no code/output give 0, step1 max 3, step2 max 2.5, step3 max 2.5, step4 max 2, total max 10'''
    resp = client.models.generate_content(model=MODEL, contents=prompt,
        config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json'))
    return json.loads(resp.text)


def append_result(info, grade, url):
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    with open(SCORES_CSV,'a',encoding='utf-8',newline='') as f:
        csv.writer(f).writerow([ts,info['student_name'],info['student_id'],info['phone'],info['line_id'],
            grade['step1_score'],grade['step2_score'],grade['step3_score'],grade['step4_score'],
            grade['total_score'],grade.get('is_ai_suspected',False),grade['overall_feedback'],url])
    with open(SCORES_JSON,'r') as f: all_r = json.load(f)
    grade.update({'timestamp':ts,'github_url':url,**info})
    all_r.append(grade)
    with open(SCORES_JSON,'w',encoding='utf-8') as f: json.dump(all_r,f,ensure_ascii=False,indent=2)
    print(f'💾 บันทึกแล้ว ({len(all_r)} คน)')  # "💾 Saved ({len(all_r)} people)"

print('✅ ฟังก์ชันพร้อม')  # "✅ Functions ready"

---
## ▶️ Grade homework (one at a time)

In [ ]:
# ============================================================
# 👇 Paste GitHub URL
# ============================================================
GITHUB_URL = ''
# ============================================================

assert GITHUB_URL, '❌ กรุณาวาง URL!'  # "❌ Please paste the URL!"

info, content = fetch_notebook(GITHUB_URL)
print(f'👤 {info["student_name"]} ({info["student_id"]})')

# Check duplicate
already = False
if os.path.exists(SCORES_JSON):
    with open(SCORES_JSON) as f:
        for p in json.load(f):
            if p.get('student_id')==info['student_id']:
                already=True; print(f'⚠️ เคยตรวจแล้ว (คะแนน: {p.get("total_score")})')  # "⚠️ Already graded before (score: {p.get("
                break
if already:
    ow = input('🔄 ตรวจใหม่? (y/n): ').strip().lower()  # "🔄 Grade again? (y/n): "
    if ow != 'y': raise SystemExit('⏭️ ข้าม')  # "⏭️ Skip"
    with open(SCORES_JSON) as f: prev=json.load(f)
    prev=[p for p in prev if p.get('student_id')!=info['student_id']]
    with open(SCORES_JSON,'w',encoding='utf-8') as f: json.dump(prev,f,ensure_ascii=False,indent=2)
    rows=[]
    with open(SCORES_CSV,'r',encoding='utf-8') as f: rows=list(csv.reader(f))
    with open(SCORES_CSV,'w',encoding='utf-8',newline='') as f:
        w=csv.writer(f)
        for r in rows:
            if len(r)>2 and r[2]==info['student_id']: continue
            w.writerow(r)

expected = generate_expected_day3(info['student_id'])
print(f'🔑 เฉลย: topics={expected["topics"]}')  # "🔑 Answer key: topics={expected["
print(f'🤖 ตรวจด้วย {MODEL}...')  # "🤖 Grading with {MODEL}..."
grade = grade_day3(info['student_id'], content, expected)

print(f'\n{"="*60}')
print(f'📊 ผลการตรวจ: {info["student_name"]} ({info["student_id"]})')  # "📊 Grading result: {info["
print(f'{"="*60}')
print(f'  Step 1 (RAG Pipeline): {grade["step1_score"]}/3    — {grade["step1_feedback"]}')
print(f'  Step 2 (Evaluation):   {grade["step2_score"]}/2.5  — {grade["step2_feedback"]}')
print(f'  Step 3 (Optimization): {grade["step3_score"]}/2.5  — {grade["step3_feedback"]}')
print(f'  Step 4 (Agent+Test):   {grade["step4_score"]}/2    — {grade["step4_feedback"]}')
print(f'  {"─"*56}')
print(f'  🏆 คะแนนรวม: {grade["total_score"]}/10')  # "  🏆 Total score: {grade["
if grade.get('is_ai_suspected'): print(f'  ⚠️ AI: {grade["ai_reason"]}')

append_result(info, grade, GITHUB_URL)

## 📊 View all scores

In [ ]:
with open(SCORES_CSV, 'r', encoding='utf-8') as f:
    rows = list(csv.reader(f))[1:]
print(f'📊 ตรวจแล้ว {len(rows)} คน\n')  # "📊 Graded {len(rows)} people\n"
print(f'{"#":>3} {"ชื่อ":<20} {"รหัส":<12} {"S1":>4} {"S2":>4} {"S3":>4} {"S4":>4} {"รวม":>5} {"AI?":>4}')  # ":>3} {"
print('='*72)
for i,r in enumerate(rows,1):
    ai='⚠️' if r[10]=='True' else '✅'
    print(f'{i:>3} {r[1]:<20} {r[2]:<12} {r[5]:>4} {r[6]:>4} {r[7]:>4} {r[8]:>4} {r[9]:>5} {ai:>4}')
if rows:
    scores=[float(r[9]) for r in rows]
    print(f'\n📈 min={min(scores):.1f}, max={max(scores):.1f}, avg={sum(scores)/len(scores):.1f}')

## 💾 Download

In [ ]:
try:
    from google.colab import files
    files.download(SCORES_CSV)
except:
    print(f'📄 {os.path.abspath(SCORES_CSV)}')